In [ ]:
# !pip install --upgrade kaggle
# !pip uninstall numpy
# !pip install "numpy==1.26.4"
# !pip install ray
# !pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/18.2 MB 23.3 kB/s eta 0:11:21
^C
ERROR: Operation cancelled by user


In [1]:
import torch, numpy as np, ray, kaggle
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print("CUDA:", torch.cuda.is_available())
print("ray:", ray.__version__)
print("kaggle", kaggle.__version__)

/home/yonataba/.conda/envs/RecSys/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-13 12:42:56,634	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Torch: 2.9.1+cu128
NumPy: 1.26.4
CUDA: True
ray: 2.52.1


AttributeError: module 'kaggle' has no attribute '__version__'

In [17]:
!kaggle competitions download -c recsys-course-2025

 77%|█████████████████████████████▉         | 46.0M/59.9M [00:00<00:00, 217MB/s]
100%|███████████████████████████████████████| 59.9M/59.9M [00:00<00:00, 209MB/s]


In [18]:
!unzip -o recsys-course-2025.zip

Archive:  recsys-course-2025.zip
  inflating: All_Beauty.train.csv    
  inflating: All_Beauty.valid.csv    
  inflating: meta_All_Beauty.jsonl   
  inflating: test.csv                


In [2]:
import pandas as pd

df_train = pd.read_csv('All_Beauty.train.csv')
df_train.head(3)

,user_id,parent_asin,rating,timestamp,history
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855070,NaN
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728923,B081TJ8YS3
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,1589665266052,NaN


In [ ]:
df_test = pd.read_csv('test.csv')
df_test.head(3)

,Unnamed: 0,id,history
0,0,0,B0020MKBNW B082FLP15V B00946HGLW
1,1,1,B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
2,2,2,B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...


: 

In [7]:
!git clone https://github.com/RUCAIBox/RecBole.git
!git -C ./RecBole/ checkout v1.2.1
!sed -i '/ray>=1.13.0/d' ./RecBole/setup.py
!pip install -e ./RecBole/

Cloning into 'RecBole'...
remote: Enumerating objects: 23604, done.
remote: Total 23604 (delta 0), reused 0 (delta 0), pack-reused 23604 (from 1)
Receiving objects: 100% (23604/23604), 19.47 MiB | 29.45 MiB/s, done.
Resolving deltas: 100% (15703/15703), done.
Note: switching to 'v1.2.1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 9a6f63d8 Merge pull request #2143 from Fotiligner/master
Obtaining file:///content/RecBole
  Preparing metadata (setup.py) ... done
  Running setup.py develop for recbole


In [1]:
import recbole
print(recbole.__file__)


ModuleNotFoundError: No module named 'recbole'

In [5]:
import pandas as pd
import os

# Load your CSVs (ONLY train and valid for now)
train = pd.read_csv("All_Beauty.train.csv")
valid = pd.read_csv("All_Beauty.valid.csv")

# Mark splits
train["split"] = "train"
valid["split"] = "valid"

# Merge
df = pd.concat([train, valid], ignore_index=True)

# Rename for RecBole
df = df.rename(columns={'parent_asin': 'item_id'})

# Keep only the fields RecBole should see
df = df[['user_id', 'item_id', 'rating', 'timestamp', 'history', 'split']]

# Clean history field
df["history"] = df["history"].fillna("").astype(str)

def fix_history(h):
    h = str(h).strip()
    if h.startswith("[") and h.endswith("]"):
        try:
            lst = eval(h)
            if isinstance(lst, list):
                return " ".join(str(x) for x in lst)
        except:
            return ""
    return h

df["history"] = df["history"].apply(fix_history)

# Create RecBole directory
os.makedirs("dataset/All_Beauty", exist_ok=True)

# Correct header for RecBole .inter
header = (
    "user_id:token\t"
    "item_id:token\t"
    "rating:float\t"
    "timestamp:float\t"
    "history:token_seq\t"
    "split:token"
)

# Write .inter file
with open("dataset/All_Beauty/All_Beauty.inter", "w") as f:
    f.write(header + "\n")
    df.to_csv(f, sep="\t", index=False, header=False)

print("✔ Rebuilt All_Beauty.inter WITHOUT test split in correct RecBole format!")


✔ Rebuilt All_Beauty.inter WITHOUT test split in correct RecBole format!


In [18]:
import json
import pandas as pd
import os

ROOT = "dataset/All_Beauty"
os.makedirs(ROOT, exist_ok=True)

print("🚀 Building UniSRec dataset…")

# -------------------------------
# 1. Load train + valid
# -------------------------------
train = pd.read_csv("All_Beauty.train.csv")
valid = pd.read_csv("All_Beauty.valid.csv")

print("Train columns:", train.columns.tolist())
print("Valid columns:", valid.columns.tolist())

inter = pd.concat([train, valid], ignore_index=True)
inter = inter.sort_values(["user_id", "timestamp"])

# -------------------------------
# 2. Build user2seq.json
# -------------------------------
user2seq = {}

for uid, g in inter.groupby("user_id"):
    seq = g.sort_values("timestamp")["parent_asin"].tolist()
    if len(seq) > 0:
        user2seq[uid] = seq

with open(f"{ROOT}/user2seq.json", "w") as f:
    json.dump(user2seq, f)

print(f"✔ user2seq.json written: {len(user2seq)} users")

# -------------------------------
# 3. Split into UniSRec train/valid/test
# -------------------------------
def write_txt(path, mapping):
    with open(path, "w") as f:
        for uid, seq in mapping.items():
            f.write(uid + "\t" + " ".join(seq) + "\n")

# For UniSRec, we typically use the SAME sequences for all three
# because train/valid/test is done by RecBole evaluators
write_txt(f"{ROOT}/train.txt", user2seq)
write_txt(f"{ROOT}/valid.txt", user2seq)
write_txt(f"{ROOT}/test.txt", user2seq)

print("✔ train.txt / valid.txt / test.txt written")

# -------------------------------
# 4. Build item_text.json from meta
# -------------------------------
item_text = {}
bad = 0

with open("meta_All_Beauty.jsonl") as f:
    for line in f:
        try:
            obj = json.loads(line)
            asin = obj["parent_asin"]
            title = obj.get("title", "")
            desc  = " ".join(obj.get("description", []))
            text  = (title + " " + desc).strip()
            if len(text) > 0:
                item_text[asin] = text
        except:
            bad += 1

with open(f"{ROOT}/item_text.json", "w") as f:
    json.dump(item_text, f)

print(f"✔ item_text.json written: {len(item_text)} items, {bad} bad lines")

print("\n🎉 UniSRec dataset preparation complete!")


🚀 Building UniSRec dataset…
Train columns: ['user_id', 'parent_asin', 'rating', 'timestamp', 'history']
Valid columns: ['user_id', 'parent_asin', 'rating', 'timestamp', 'history']
✔ user2seq.json written: 597809 users
✔ train.txt / valid.txt / test.txt written
✔ item_text.json written: 112579 items, 0 bad lines

🎉 UniSRec dataset preparation complete!


In [8]:
import json

item_text = {}

with open("meta_All_Beauty.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)

        asin = data.get("parent_asin")
        title = data.get("title", "")
        desc = " ".join(data.get("description", []))
        features = " ".join(data.get("features", []))

        # combine text fields
        content = f"{title} {desc} {features}".strip()

        if asin and content:
            item_text[asin] = content

# write UniSRec format file
with open("dataset/item_text.txt", "w") as out:
    for asin, text in item_text.items():
        out.write(f"{asin}\t{text}\n")

print("✔ Created dataset/item_text.txt for UniSRec")


✔ Created dataset/item_text.txt for UniSRec


In [11]:
import json
import re
import pandas as pd

input_jsonl = "meta_All_Beauty.jsonl"
output_txt = "dataset/item_text.txt"

def clean_text(x):
    if not x:
        return ""
    if isinstance(x, list):
        x = " ".join(x)
    x = re.sub(r'\s+', ' ', x)
    return x.strip()

rows = []
skipped = 0

with open(input_jsonl, "r") as f:
    for line in f:
        obj = json.loads(line)

        asin = obj.get("parent_asin")
        if not asin:
            skipped += 1
            continue

        title = clean_text(obj.get("title", ""))
        features = clean_text(obj.get("features", []))
        description = clean_text(obj.get("description", []))

        text = f"{title} {features} {description}".strip()

        if text == "":
            skipped += 1
            continue

        rows.append([asin, text])

df = pd.DataFrame(rows, columns=["item_id", "item_text"])
df.to_csv(output_txt, sep="\t", index=False)

print("✔ Clean item_text.txt generated!")
print(f"✔ Total items with text: {len(df)}")
print(f"⚠ Skipped items: {skipped}")


✔ Clean item_text.txt generated!
✔ Total items with text: 112581
⚠ Skipped items: 9


In [12]:
import pandas as pd

df = pd.read_csv("dataset/item_text.txt", sep="\t", header=0)
df.to_csv("dataset/All_Beauty/All_Beauty.item", sep="\t", index=False)

print("✔ All_Beauty.item recreated cleanly!")


✔ All_Beauty.item recreated cleanly!


In [13]:
yaml_unisrec = """\
field_separator: "\\t"
seq_separator: " "

data_path: "./dataset"

USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
RATING_FIELD: rating
TIME_FIELD: timestamp

load_col:
  inter: [user_id, item_id, rating, timestamp, history, split]
  item: [item_id, item_text]

history_field: history
split_field: split

# ----------------------
# UniSRec model settings
# ----------------------
model: UniSRec

TEXT_FIELD: item_text
text_col:
  item: [item_text]

# UniSRec-specific configuration
embedding_size: 64
hidden_size: 64
dropout_prob: 0.1
augment_threshold: 0.1

# Text embeddings (for Amazon datasets)
embedding_source: glove.6B.100d

# Training configuration
loss_type: CE
train_neg_sample_args: null
eval_neg_sample_args: null

epochs: 200
train_batch_size: 4096
learning_rate: 0.001

metrics: [Recall, NDCG]
topk: [10]
valid_metric: "Recall@10"
"""

with open("All_Beauty.yaml", "w") as f:
    f.write(yaml_unisrec)

print("✔ UniSRec YAML written!")


✔ UniSRec YAML written!


In [45]:
import sys
sys.path.append('RecBole')  # or wherever you cloned it

In [41]:
import os
import re

root = "RecBole"

def patch_all_torch_loads():
    for subdir, _, files in os.walk(root):
        for file in files:
            if file.endswith(".py"):
                path = os.path.join(subdir, file)
                with open(path, "r") as f:
                    code = f.read()

                # Patch ONLY torch.load(...) that does not already specify weights_only=
                new_code, count = re.subn(
                    r"torch\.load\(([^),]+)(\))",
                    r"torch.load(\1, weights_only=False\2",
                    code
                )

                if count > 0:
                    with open(path, "w") as f:
                        f.write(new_code)
                    print(f"Patched {count} occurrence(s) in {path}")

patch_all_torch_loads()

print("✔ All torch.load() calls patched with weights_only=False")


✔ All torch.load() calls patched with weights_only=False


In [30]:
import sys, subprocess
print("python:", sys.executable)
print("pip:", subprocess.check_output("which pip", shell=True).decode())
print(subprocess.check_output("pip --version", shell=True).decode())


python: /home/yonataba/.conda/envs/RecSys/bin/python
pip: /home/yonataba/.conda/envs/RecSys/bin/pip

pip 25.3 from /home/yonataba/.conda/envs/RecSys/lib/python3.10/site-packages/pip (python 3.10)



In [17]:
import json
import pandas as pd
import os

ROOT = "./dataset/All_Beauty"
os.makedirs(ROOT, exist_ok=True)

print("🚀 Building UniSRec dataset...")

# ----------------------------------------------------
# 1. Load interactions (train + valid)
# ----------------------------------------------------
train = pd.read_csv("All_Beauty.train.csv")
valid = pd.read_csv("All_Beauty.valid.csv")

inter = pd.concat([train, valid], ignore_index=True)
inter = inter.sort_values(["user_id", "timestamp"])

# ----------------------------------------------------
# 2. Build user → sequence mapping
# ----------------------------------------------------
user2seq = {}

for uid, group in inter.groupby("user_id"):
    seq = group.sort_values("timestamp")["item_id"].tolist()
    user2seq[uid] = seq

with open(f"{ROOT}/user2seq.json", "w") as f:
    json.dump(user2seq, f)

print(f"✔ user2seq.json written ({len(user2seq)} users)")


# ----------------------------------------------------
# 3. Write train/valid/test in UniSRec format
# Format: user_id TAB item1 item2 item3 ...
# ----------------------------------------------------

def write_seq_file(df, fname):
    with open(f"{ROOT}/{fname}", "w") as f:
        for uid, seq in user2seq.items():
            if len(seq) == 0:
                continue
            f.write(uid + "\t" + " ".join(seq) + "\n")

write_seq_file(train, "train.txt")
write_seq_file(valid, "valid.txt")
write_seq_file(pd.read_csv("test.csv"), "test.txt")

print("✔ train.txt / valid.txt / test.txt written")


# ----------------------------------------------------
# 4. Build item_text.json using meta_All_Beauty.jsonl
# ----------------------------------------------------
item_text = {}
bad = 0

with open("meta_All_Beauty.jsonl") as f:
    for line in f:
        try:
            obj = json.loads(line)
            asin = obj["parent_asin"]
            title = obj.get("title", "")
            desc = " ".join(obj.get("description", []))

            text = (title + " " + desc).strip()

            if len(text) > 0:
                item_text[asin] = text
        except:
            bad += 1

with open(f"{ROOT}/item_text.json", "w") as f:
    json.dump(item_text, f)

print(f"✔ item_text.json written ({len(item_text)} items, {bad} bad lines)")

print("\n🎉 UniSRec dataset complete!")


🚀 Building UniSRec dataset...


KeyError: 'item_id'

In [16]:
from recbole.quick_start import run_recbole
run_recbole(model="UniSRec", dataset="All_Beauty", config_file_list=["All_Beauty.yaml"])


ValueError: `model_name` [UniSRec] is not the name of an existing model.

In [27]:
import os

for root, dirs, files in os.walk("./", topdown=True):
    if "pth" in str(files).lower():
        print(root, files)


./saved ['SASRec-Dec-08-2025_22-35-24.pth']


In [2]:
import torch
from recbole.model.sequential_recommender import SASRec
from recbole.data.utils import create_dataset, data_preparation
from recbole.config import Config

# === 1) Load config ===
config = Config(
    model="SASRec",
    dataset="All_Beauty",
    config_file_list=["All_Beauty.yaml"]
)

# === 2) Load dataset ===
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

# === 3) Load trained model ===
CHECKPOINT = "saved/SASRec-Dec-08-2025_22-35-24.pth"   # ← you will paste the path after next step

checkpoint = torch.load(CHECKPOINT, map_location=config["device"], weights_only=False)

model = SASRec(config, train_data.dataset).to(config["device"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()

print("✔ Model successfully loaded")


/home/yonataba/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
/home/yonataba/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:1217: FutureWarning: using <built-in function len> in Series.agg cannot aggregate and has been deprecated. Use Series.transform to keep behavior unchanged.
  split_point = np.cumsum(feat[field].agg(len))[:-1]
/home/yonataba/.conda/envs/RecSys/lib/pyt

✔ Model successfully loaded


In [12]:
import pandas as pd

test_df = pd.read_csv("test.csv")
print(test_df.head())


   Unnamed: 0  id                                            history
0           0   0                   B0020MKBNW B082FLP15V B00946HGLW
1           1   1  B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
2           2   2  B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
3           3   3  B00N6WHTRG B00NNKWDI6 B00MDKICPK B010B0S67C B0...
4           4   4  B07FM69672 B07J3GH1W1 B07F8PCLDV B071JMGPTH B0...


In [4]:
import torch
import pandas as pd
from tqdm import tqdm
from recbole.data import create_dataset, data_preparation
from recbole.data.interaction import Interaction

# ------------------------------
# 1. Load dataset (RecBole)
# ------------------------------

uid_field = dataset.uid_field
iid_field = dataset.iid_field

token2id_item = dataset.field2token_id[iid_field]
id2token_item = lambda x: dataset.id2token(iid_field, x)

df_test = pd.read_csv("test.csv")

# SASRec sequence fields
item_seq_field = model.ITEM_SEQ       # "item_id_list"
item_seq_len_field = model.ITEM_SEQ_LEN  # "item_length"
max_len = config['MAX_ITEM_LIST_LENGTH']

device = model.device
top_k = 10

model.eval()

print(f"🔥 Running batched inference on device: {device}")
print(f"🧠 Max sequence length: {max_len}")

# ------------------------------------------
# 2. Preprocess all histories
# ------------------------------------------
seqs = []
seq_lens = []
seen_sets = []
session_ids = df_test["id"].tolist()

for _, row in df_test.iterrows():
    history_asins = row["history"].split()

    internal_hist = [token2id_item[a] for a in history_asins if a in token2id_item]
    if len(internal_hist) == 0:
        internal_hist = [0]

    seen_sets.append(set(internal_hist))

    seq = internal_hist[-max_len:]
    seq_len = len(seq)
    padding = [0] * (max_len - seq_len)
    seq_padded = seq + padding

    seqs.append(seq_padded)
    seq_lens.append(seq_len)

# convert to GPU tensors
seqs = torch.tensor(seqs, dtype=torch.long).to(device)
seq_lens = torch.tensor(seq_lens, dtype=torch.long).to(device)
uids = torch.zeros(len(df_test), dtype=torch.long).to(device)  # dummy

# ------------------------------------------
# 3. Predict in batches
# ------------------------------------------
batch_size = 512
all_scores = []

print("🚀 Predicting in batches...")

with torch.no_grad():
    for i in tqdm(range(0, len(df_test), batch_size)):
        b_seq = seqs[i:i+batch_size]
        b_len = seq_lens[i:i+batch_size]
        b_uid = uids[i:i+batch_size]

        interaction = Interaction({
            uid_field: b_uid,
            item_seq_field: b_seq,
            item_seq_len_field: b_len
        })

        scores = model.full_sort_predict(interaction)
        all_scores.append(scores)

scores = torch.cat(all_scores, dim=0)

# ------------------------------------------
# 4. Mask seen items
# ------------------------------------------
for idx, seen in enumerate(seen_sets):
    if seen:
        scores[idx, list(seen)] = -1e9

# ------------------------------------------
# 5. Top-K and convert to ASINs
# ------------------------------------------
topk_indices = torch.topk(scores, top_k, dim=1).indices.cpu().tolist()

# ------------------------------------------
# 6. Convert to required Kaggle format
# ------------------------------------------
rows = []

for session_id, internal_items in zip(session_ids, topk_indices):
    asin_items = [id2token_item(i) for i in internal_items]
    row = {"id": session_id}
    for j in range(10):
        row[f"rec{j+1}"] = asin_items[j]
    rows.append(row)

submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

print("✔ submission.csv written in correct format!")
submission.head()


/home/yonataba/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
/home/yonataba/.conda/envs/RecSys/lib/python3.10/site-packages/recbole/data/dataset/dataset.py:1217: FutureWarning: using <built-in function len> in Series.agg cannot aggregate and has been deprecated. Use Series.transform to keep behavior unchanged.
  split_point = np.cumsum(feat[field].agg(len))[:-1]
/home/yonataba/.conda/envs/RecSys/lib/pyt

🔥 Running batched inference on device: cuda
🧠 Max sequence length: 50
🚀 Predicting in batches...


100%|██████████| 10/10 [00:00<00:00, 675.06it/s]


✔ submission.csv written in correct format!


,id,rec1,rec2,rec3,rec4,rec5,rec6,rec7,rec8,rec9,rec10
0,0,B07DQT7R5J,B000NJMSIU,B07DHLPZLY,B01FYUYIWG,B07BHL8QYY,B0180VBNWO,B0768DD291,B00UMR1C0S,B010B5ZE4U,B00DOLG9TI
1,1,B07ZXM2TQ1,B09535T1FN,B00DZQOAVG,B07NVWT3RF,B08HMQJDH7,B0167JZBRY,B07CMGQ25F,B07DN6VRP5,B07DWBFTW2,B091MT712J
2,2,B07ZXM2TQ1,B09535T1FN,B00DZQOAVG,B07NVWT3RF,B08HMQJDH7,B0167JZBRY,B07CMGQ25F,B07DN6VRP5,B07DWBFTW2,B091MT712J
3,3,B07ZXM2TQ1,B09535T1FN,B00DZQOAVG,B07NVWT3RF,B08HMQJDH7,B0167JZBRY,B07CMGQ25F,B07DN6VRP5,B07DWBFTW2,B091MT712J
4,4,B01J6I6MJY,B074NYS3BX,B01ENDFXJW,B019BCAMT6,B07H53GCV2,B07B2Q84HD,B00JJYQ4QC,B08PSS11NS,B07R8V46J4,B00GGU1LJM


In [6]:
import subprocess

competition = "recsys-course-2025"
file_path = "submission.csv"
message = "First attempt"

cmd = [
    "kaggle", "competitions", "submit",
    "-c", competition,
    "-f", file_path,
    "-m", message
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


401 Client Error: Unauthorized for url: https://www.kaggle.com/api/v1/competitions/submission-url




In [7]:
import kaggle
kaggle.api.authenticate()
print("Authenticated!")


Authenticated!
